In [2]:
import os
import re
import sys
import json
import torch
import pickle
import contextlib
import numpy as np
from tqdm import tqdm
from typing import List, Tuple, Dict

from qiskit_aer import AerSimulator
from pytket.extensions.qiskit.backends.aer import AerBackend
# from qiskit.providers.aer import AerSimulator
# from pytket.extensions.qiskit import AerBackend


from lambeq.backend.grammar import Diagram, Id
from lambeq import (
    AtomicType,
    IQPAnsatz,
    RemoveCupsRewriter,
    SimpleRewriteRule,
    Rewriter,
    UnifyCodomainRewriter,
    DepCCGParser
)

# import depccg
# from lambeq import ( CCGParser, CCGTree, CCGRuleUseError, CCGRule, CCGType,
#                     CCGBankParseError, CCGBankParser, DepCCGParseError )


/home/green/QNLPModelTraining/qnlp_3_10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
import logging
logging.getLogger("allennlp").setLevel(logging.WARNING)
logging.getLogger("depccg").setLevel(logging.WARNING)
parser = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

In [10]:
def get_deep_type(obj):
    if isinstance(obj, list):
        # We look at the unique types inside the list to keep it readable
        inner_types = {get_deep_type(item) for item in obj}
        return f"List[{' | '.join(sorted(inner_types))}]"

    elif isinstance(obj, dict):
        # We summarize the types of all keys and all values
        key_types = {get_deep_type(k) for k in obj.keys()}
        val_types = {get_deep_type(v) for v in obj.values()}
        return f"Dict[{' | '.join(sorted(key_types))}, {' | '.join(sorted(val_types))}]"

    else:
        # Return the class name (e.g., 'Diagram' or 'str')
        return type(obj).__name__
    
from collections.abc import Mapping, Sequence

def get_deep_shape(obj, level=0):
    indent = "  " * level

    # 1. Atomic types (Strings/Bytes) - Removed len() for brevity
    if isinstance(obj, (str, bytes)):
        return f"{indent}str"

    # 2. Sequences (Lists, Tuples, etc.)
    elif isinstance(obj, Sequence):
        if not obj:
            return f"{indent}{type(obj).__name__}(len=0)"

        header = f"{indent}{type(obj).__name__}(len={len(obj)})"
        
        # Calculate shapes of all children to check for uniformity
        child_shapes = [get_deep_shape(item, level + 1).lstrip() for item in obj]
        unique_shapes = sorted(list(set(child_shapes)))

        if len(unique_shapes) == 1:
            # All items are identical structure (e.g., all are 'str')
            return f"{header}\n{indent}  [*] -> {unique_shapes[0]}"
        else:
            # Items differ; list them individually
            # Note: For massive lists with mixed types, you might want to 
            # only show the first few, but here we show all as requested.
            lines = [header]
            for i, shape in enumerate(child_shapes):
                lines.append(f"{indent}  [{i}] -> {shape}")
            return "\n".join(lines)

    # 3. Dictionaries
    elif isinstance(obj, Mapping):
        if not obj:
            return f"{indent}dict(len=0)"

        lines = [f"{indent}dict(len={len(obj)})"]
        for key, value in obj.items():
            child = get_deep_shape(value, level + 1).lstrip()
            lines.append(f"{indent}  key={repr(key)} -> {child}")
        return "\n".join(lines)

    # 4. Base objects
    else:
        return f"{indent}{type(obj).__name__}"

def find_mismatches(data_a, data_b):
    # 1. Check if the outer lists are even the same length
    if len(data_a) != len(data_b):
        print(
            f"❌ [CRITICAL] Outer List Length Mismatch: List A={len(data_a)}, List B={len(data_b)}"
        )

    # Iterate through the top-level list
    for i, (dict_a, dict_b) in enumerate(zip(data_a, data_b)):
        # Check if the keys in the dictionaries match
        keys_a = set(dict_a.keys())
        keys_b = set(dict_b.keys())

        if keys_a != keys_b:
            print(f"❌ [Index {i}] Key Mismatch:")
            print(f"   Keys only in A: {keys_a - keys_b}")
            print(f"   Keys only in B: {keys_b - keys_a}")
            continue  # Skip to next list item if keys don't match

        # Check values for each key
        for key in keys_a:
            list_a = dict_a[key]
            list_b = dict_b[key]

            # 2. Check lengths of the lists inside the dictionary
            if len(list_a) != len(list_b):
                print(f"❌ [Index {i}][Key: '{key}'] Inner List Length Mismatch:")
                print(f"   Length A: {len(list_a)}")
                print(f"   Length B: {len(list_b)}")

            # 3. Check individual elements inside those lists
            # This handles List[Diagram], List[List[int]], and List[str]
            for j, (val_a, val_b) in enumerate(zip(list_a, list_b)):
                if val_a != val_b:
                    print(
                        f"❌ [Index {i}][Key: '{key}'][Inner Index {j}] Content Mismatch!"
                    )
                    print(f"   Type A: {type(val_a).__name__}")
                    print(f"   Type B: {type(val_b).__name__}")

                    # If they are small (like List[int] or str), print the actual value
                    if not hasattr(
                        val_a, "draw"
                    ):  # Don't print full Diagrams, they are too big
                        print(f"   Value A: {val_a}")
                        print(f"   Value B: {val_b}")
                    else:
                        print(
                            f"   (Diagram content differs - possibly different boxes or wires)"
                        )

def diagnose_variable(variable):
    # print("TOP LEVEL LENGTH:", len(variable) if hasattr(variable, '__len__') else "N/A")
    print("DEEP TYPE:")
    print(get_deep_type(variable))
    print("\nDEEP SHAPE:")
    print(get_deep_shape(variable))


In [ ]:
#################################################
#### Mainly to test if GPU works with:       ####
#### sim.run(tk_circ, n_shots=1) and         ####
#### backend.run_circuit(tk_circ, n_shots=1) ####
#################################################


In [26]:
def encode_debug(dataset):
    encoded_data, errors = [], []
    # errs1, errs2, errs3, errs4 = [], [], [], []
    # combined_dataset = combine_articles(dataset)
    # diagnose_variable(combined_dataset)

    for i, data_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
        with open(os.devnull, 'w') as devnull, \
         contextlib.redirect_stdout(devnull), \
         contextlib.redirect_stderr(devnull):

    # for i, dict in enumerate(dataset):
            text_sentences = data_dict["text_sentences"]
            labels         = data_dict["labels"]

            sentences_simplified = sentence_simplify(text_sentences)
            diagrams, remove     = sent2diagrams(sentences_simplified)
            diagrams             = remove_by_idx(diagrams, remove)
            text_sentences       = remove_by_idx(text_sentences, remove)
            labels               = remove_by_idx(labels, remove)

            normalized_diagrams, remove, errs2 = normalize(diagrams)
            text_sentences                     = remove_by_idx(text_sentences, remove)
            labels                             = remove_by_idx(labels, remove)

            circuits, remove, errs3 = quantum_encode(normalized_diagrams)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            circuits, remove, errs4 = will_train(circuits)
            text_sentences          = remove_by_idx(text_sentences, remove)
            labels                  = remove_by_idx(labels, remove)

            encoded_data.append(
                {
                    "circuits": circuits,
                    "labels": labels,
                    "original_text_sentences": text_sentences,
                }
            )

    errors += errs2 + errs3 + errs4 # + errs1 sent2diagrams has suppress_exceptions=True, so instead of errors, it returns None

    return encoded_data, errors


def combine_articles(dataset) -> List[Dict]:
    combined_text_sentences, combined_labels = [], []

    for article in dataset:
        # zip(combined, article["text_sentences"]) ####### paklaust gpt kam naudojamas zip ir kodel jis cia neveikia, i.e. kaip jis cia tiksliai veikia?
        # zip(labels, article["labels"])

        combined_text_sentences += article["text_sentences"]
        combined_labels += article["labels"]

    combined_dataset = [{
        "text_sentences": combined_text_sentences,
        "labels": combined_labels,
    }]

    return combined_dataset

def combine_n_articles(dataset, n_to_merge: int = 3) -> List[Dict]:
    combined_dataset = []
    
    for i in range(0, len(dataset), n_to_merge):
        chunk = dataset[i : i + n_to_merge]
        combined_dataset.extend(combine_articles(chunk))
        
    return combined_dataset


def load_PreSumm_pts(left=0, right=144, ds_purpose = "train", calculate_articles: bool = False) -> List[Dict]:
    raw_ds = []
    k = left
    n_articles = []
    
    while k < right:
        print(f"load_PreSumm_pts_{k}")
        file_path = f"Dataset/Raw/cnn_dailymail/_PreSumm/cnndm.{ds_purpose}.{k}.bert.pt"
        loaded_data = torch.load(file_path)

        for i, file_element in enumerate(loaded_data):

            quantum_state_distribution_labels = []
            for label in file_element["src_sent_labels"]:
                if label:
                    quantum_state_distribution_labels.append([0,1])
                else:
                    quantum_state_distribution_labels.append([1,0])

            raw_ds.append(
                {
                    "text_sentences": file_element["src_txt"],
                    # "org_labels": line["src_sent_labels"],
                    "labels": quantum_state_distribution_labels
                }
            )
        n_articles.append(len(loaded_data))
        k += 1
        
    return (raw_ds, n_articles) if calculate_articles else raw_ds

HIGHEST_FILENAME_NUMBER = 8

def read_PreSum_multiple(amount: int = 4, dataset_purpose: str = "train"):
    left = 0
    right = amount
    loaded_combined_dataset = []

    while left < HIGHEST_FILENAME_NUMBER:
        print(f"read_PreSumm_multiple_{left}")
        loaded_combined_dataset.append(
            combine_articles( load_PreSumm_pts(left, right, dataset_purpose) )
        )
        left = right
        right = min(HIGHEST_FILENAME_NUMBER, left + amount)

    return loaded_combined_dataset


In [12]:
_CLEAN_REGEX = re.compile(r"[^\w\s']")
# parser    = DepCCGParser(model='elmo', device=0) # device=:  -1 == CPU | 0 == GPU | 1 == second GPU

ansatz    = IQPAnsatz(
    {AtomicType.SENTENCE: 1,
     AtomicType.NOUN:     1,
     AtomicType.PREPOSITIONAL_PHRASE: 0},
    n_layers=2, n_single_qubit_params=3
)

def create_rewriter():
    # Rule to delete conjunction boxes (“and”, “but”) # just the wire, no box
    conj_rule = SimpleRewriteRule(cod=AtomicType.CONJUNCTION, template=Id(AtomicType.CONJUNCTION))

    # Rule to delete punctuation boxes (commas, quotes, dashes)
    punc_rule = SimpleRewriteRule(cod=AtomicType.PUNCTUATION, template=Id(AtomicType.PUNCTUATION))

    # remove_pp2 = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.SENTENCE))
    remove_pp = SimpleRewriteRule(cod=AtomicType.PREPOSITIONAL_PHRASE, template=Id(AtomicType.NOUN))

    rewriter = Rewriter(
        [
            'coordination', 'determiner',
            'postadverb', 'preadverb',
            'connector', 'auxiliary',
            'prepositional_phrase',
            'subject_rel_pronoun',
            'object_rel_pronoun',
        ]
    )
    rewriter.add_rules(remove_pp, punc_rule, conj_rule)
    return rewriter

rewriter = create_rewriter()
remove_cups = RemoveCupsRewriter()
unify = UnifyCodomainRewriter(output_type=AtomicType.SENTENCE)

In [13]:
simulator = AerSimulator(
    method="statevector", device="GPU",
    precision="single",         # 32-bit float for ~2× speedup on large statevectors
    cuStateVec_enable=True,     # turn on NVIDIA cuStateVec kernels
    batched_shots_gpu=True,     # batch thousands of shots very efficiently on GPU
    batched_shots_gpu_max_qubits=40,
    num_threads_per_device=2    # limit CPU threads per GPU to reduce overhead
)

backend = AerBackend(noise_model=None, simulation_method="statevector")
backend._qiskit_backend = simulator

comp_pass = backend.default_compilation_pass(2)

In [4]:
#############################################
# Step 2. Preprocessing and lambeq Pipeline
#############################################

def remove_by_idx(ls, remove):
    if not remove:
        return ls
    remove.sort(reverse=True)
    for n in remove:
        ls.pop(n)
    return ls

def sentence_simplify(sentences):
    return [_CLEAN_REGEX.sub("", s) for s in sentences if s is not None]

def sent2diagrams(sentences):
    none_idx = []
    diagrams = parser.sentences2diagrams(sentences, tokenised=False, suppress_exceptions=True)
    for i, diag in enumerate(diagrams):
        if diag is None:
            none_idx.append(i)
    
    print(f"Whilst parsing sentences2diagrams, lost {len(none_idx)} due to Null, out of {len(sentences)}.")

    return diagrams, none_idx

def normalize(sentence_diagrams):
    diagrams_normalized, none_idx, errs = [], [], []
    drop_rewrite = 0
    drop_cups = 0
    for i, d in enumerate(sentence_diagrams):
        try:
            d = rewriter(d)
            if (d is None):
                none_idx.append(i)
                drop_rewrite += 1
                continue

            d = remove_cups(d)
            d = d.normal_form()
            d = d.pregroup_normal_form()
            d = unify(d)

        except Exception as e:
            none_idx.append(i)
            errs.append(f"{e} ( normalize() )")
            drop_cups += 1
            continue

        diagrams_normalized.append(d)

    print(f"Dropped {drop_rewrite} diagrams in rewrite, {drop_cups} in cup removal, out of {len(sentence_diagrams)}.")

    return diagrams_normalized, none_idx, errs

def quantum_encode(diagrams: "List"):
    encoded_diagrams, remove, errs = [], [], []
    for i, diagram in enumerate(diagrams):
        try:
            circ    = ansatz(diagram)
            # circ = circ.to_tk()
            encoded_diagrams.append(circ)
        except Exception as e:
            errs.append(f"{e} ( quantum_encode() )")
            remove.append(i)

    return encoded_diagrams, remove, errs

def will_train(
    circuits,
    qubit_limit: int = 40,
    mem_limit_bytes: int = 7 * 2**30,   # 7 GiB
):

    valid, invalid_idxs, errs = [], [], []
    for idx, circ in enumerate(circuits):
        try:
            tk_circ = circ.to_tk()
            needed = 16 * (2 ** tk_circ.n_qubits)
            if needed > mem_limit_bytes:
                raise RuntimeError(f"Needs {needed} bytes > limit {mem_limit_bytes} bytes ({needed/2**20:.0f} MiB > {(mem_limit_bytes/2**20):.0f} MiB). ")

            comp_pass.apply(tk_circ)

            syms = tk_circ.free_symbols()
            if syms:
                bind_map = {s: 0.0 for s in syms}
                tk_circ.symbol_substitution(bind_map)

            simulator.run(tk_circ, n_shots=1)
            # backend.run_circuit(tk_circ, n_shots=1)

        except Exception as e:
            invalid_idxs.append(idx)
            errs.append(f"{e} ( will_train() )")
            continue

        valid.append( circ )

    return valid, invalid_idxs, errs



def preprocess_and_encode(dataset):
    encoded_data, errors = [], []
    # errs1, errs2, errs3, errs4 = [], [], [], []

    for i, ds_dict in enumerate(tqdm(dataset, desc="Filtering and Encoding dataset")):
        # with open(os.devnull, 'w') as devnull, \
        #  contextlib.redirect_stdout(devnull), \
        #  contextlib.redirect_stderr(devnull):

        text_sentences = ds_dict['text_sentences']
        labels         = ds_dict['labels']

        sentences_simplified = sentence_simplify(text_sentences)
        diagrams, remove     = sent2diagrams(sentences_simplified)
        diagrams             = remove_by_idx(diagrams, remove)
        text_sentences       = remove_by_idx(text_sentences, remove)
        labels               = remove_by_idx(labels, remove)

        normalized_diagrams, remove, errs2 = normalize(diagrams)
        text_sentences                     = remove_by_idx(text_sentences, remove)
        labels                             = remove_by_idx(labels, remove)

        circuits, remove, errs3 = quantum_encode(normalized_diagrams)
        text_sentences          = remove_by_idx(text_sentences, remove)
        labels                  = remove_by_idx(labels, remove)

        circuits, remove, errs4 = will_train(circuits)
        text_sentences          = remove_by_idx(text_sentences, remove)
        labels                  = remove_by_idx(labels, remove)

        encoded_data.append({'circuits': circuits, 'labels': labels, 'original_text_sentences': text_sentences})


        errors += errs2 + errs3 + errs4 # + errs1 sent2diagrams has suppress_exceptions=True, so instead of errors, it returns None
        print("text_sentences:", text_sentences)
        print("sentences_simplified:", len(sentences_simplified), sentences_simplified)
        print("noramlized_sentences:", len(normalized_diagrams), normalized_diagrams)
        print("circuits:", circuits)
        # print()

    return encoded_data, errors

In [ ]:
# loaded_dataset = read_PreSum_multiple()

In [8]:
ld_ds, n_articles = load_PreSumm_pts(right=1, calculate_articles=True)

load_PreSumm_pts_0


In [14]:
def print_length(dataset):
    print(sum(n_articles))
    # diagnose_variable(ld_ds)
    count = 0
    count_labels = 0
    for data in dataset:
        count += len(data["text_sentences"])
        count_labels += len(data["labels"])
    
    print(f"n_sentences: {count} | n_lables: {count_labels}")

# print_length(ld_ds[:30])

In [31]:
ld_ds_combined = combine_n_articles(ld_ds[:30], 30)
print_length(ld_ds_combined)

2001
n_sentences: 1293 | n_lables: 1293


In [ ]:
encoded_dataset, errors = encode_debug(ld_ds_combined)
# On GPU it took ~4m 36.0s to encode first 10 PreSumm articles seperately without will_train()
# On GPU it took ~14m 39.4s to encode first 30 PreSumm articles seperately without will_train()
# On GPU it took ~14m 27.1s to encode first 30 PreSumm articles (merged 2:1) without will_train()
# On GPU it took ~14m 27.6s to encode first 30 PreSumm articles (merged 3:1) without will_train()
# On GPU it took ~14m 54.1s to encode first 30 PreSumm articles (merged 6:1) without will_train()
# On GPU it took ~15m 40.0s to encode first 30 PreSumm articles (merged 10:1) without will_train()

# On GPU it took ~m .s to encode first 10 PreSumm articles seperately
# On GPU it took ~24m 30.0s to encode first 30 PreSumm articles seperately
# On GPU it took ~24m 32.6s to encode first 30 PreSumm articles (merged 2:1)
# On GPU it took ~24m 24.7s to encode first 30 PreSumm articles (merged 3:1)
# On GPU it took ~24m 46.4s to encode first 30 PreSumm articles (merged 6:1)
# On GPU it took ~25m 50.s to encode first 30 PreSumm articles (merged 10:1)
# On GPU it took ~27m 54.s to encode first 30 PreSumm articles (merged 30:1)

Filtering and Encoding dataset: 100%|██████████| 1/1 [27:53<00:00, 1673.81s/it]


In [30]:
import gc

# Delete large temporary variables
# del expensive_tensors

# Force Python to find unreferenced objects
gc.collect()

# Force the GPU to release the cached memory pool
torch.cuda.empty_cache()

In [22]:
def print_errors(errors):
    print(len(errors))
    for i, error in enumerate(errors):
        print(f"{i}: {error}")

print_errors(errors)

6
0: Diagram 0 (cod=n @ n.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ n.l @ n) does not compose with diagram 1 (dom=n @ n.l @ n @ n.l @ n @ n.r @ s @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ n.r @ n @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ n.l @ n @ n) ( normalize() )
1: Diagram 0 (cod=n @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ p.l) does not compose with diagram 1 (dom=n @ n.l @ n @ n.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ n.l @ n @ n.r @ n @ n.l @ n @ n.l @ n @ s.r @ n.r.r @ n.r @ s @ p.l @ n) ( normalize() )
2: Diagram 0 (cod=s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ s.l @ n @ n.r @ s @ p.l @ p @ n.l @ n @ n.l @ n @ n.l @ n @ s.r @ s @ s.l @ conj.l @ conj @ conj.l) does not compose with 

In [23]:
# on CPU it takes ~5m 56.5s to encode first 10 articles
# on `GPU` it takes ~4m 57.5s to encode first 10 articles

# on CPU and feeding whole articles (not single sentences) it takes ~3m 4.2s to encode first 10 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~2m 44.2s to encode first 10 articles

# on CPU and feeding multiple (40) articles it takes ~_m _._s
# on `GPU` and feeding multiple (40) articles it takes ~15m 21.3s

# on CPU and feeding whole articles (not single sentences) it takes ~m s to encode first 40 articles
# on `GPU` and feeding whole articles (not single sentences) it takes ~14m 49s to encode first 40 articles


In [ ]:
with open('Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9_somehowDifferent.pkl', 'wb') as file:
    pickle.dump(encoded_data2, file)

In [ ]:
with open('Dataset/Encoded/cnn_dailymail/cnn_dailymail_train_0_9.pkl', 'rb') as file:
    loaded_data = pickle.load(file)